# Custom Video Annotations with VideoDB

This notebook shows how to attach your own timestamped annotations to a video and make them queryable with VideoDB.

You will learn how to:

- Upload a video.
- Create custom timestamped annotation records.
- Index those records with `video.index()`.
- Query the custom annotation index with `video.query()`.
- Generate a playable stream from the matching timestamp ranges.

This pattern is useful when you already have labels, tags, chapter metadata, editorial notes, compliance markers, or human-reviewed annotations that you want to attach to video moments.

## 1. Install Dependencies

In [ ]:
!pip install -q videodb python-dotenv


## 2. Connect to VideoDB

Enter your VideoDB API key when prompted.

Enter your VideoDB API key when prompted. You can get one from the [VideoDB Console](https://console.videodb.io). Get $20 free credits. **No credit card needed**.


In [2]:
import os
import time
from getpass import getpass
from uuid import uuid4

from videodb import connect, play_stream

os.environ["VIDEO_DB_API_KEY"] = getpass("Please enter your VideoDB API Key: ")

conn = connect()
coll = conn.get_collection()

print("Connected to VideoDB successfully.")

Please enter your VideoDB API Key: ··········
Connected to VideoDB successfully.


## 3. Upload a Video

We will attach custom annotations to specific time ranges in this video.

In [3]:
video = coll.upload(url="https://www.youtube.com/watch?v=LejnTJL173Y")

print("Video ID:", video.id)
video.play()

Video ID: m-z-019f3771-57b2-71d3-ac94-a064f80e43cf


## 4. Create Custom Annotation Records

A custom temporal record is a dictionary with at least `start` and `end` timestamps. You can add your own fields such as `description`, `annotation_type`, `source`, or `confidence`.

Here we create two annotations manually. In a real application, these could come from a human review tool, an external model, an editorial system, or another database.

In [4]:
custom_annotations = [
    {
        "start": 0.0,
        "end": 10.0,
        "description": "Detective Martin is being interviewed by the police.",
        "annotation_type": "interview",
        "source": "human_review",
    },
    {
        "start": 10.0,
        "end": 100.0,
        "description": "A religious gathering where people are praying and singing.",
        "annotation_type": "religious_gathering",
        "source": "human_review",
    },
]

custom_annotations

[{'start': 0.0,
  'end': 10.0,
  'description': 'Detective Martin is being interviewed by the police.',
  'annotation_type': 'interview',
  'source': 'human_review'},
 {'start': 10.0,
  'end': 100.0,
  'description': 'A religious gathering where people are praying and singing.',
  'annotation_type': 'religious_gathering',
  'source': 'human_review'}]

## 5. Index the Custom Annotations

Use `video.index()` to make the custom timestamped records queryable.

The `fields` configuration controls how each field can be used:

- `semantic`: fields available for semantic retrieval.
- `text`: fields available as text context.
- `filter`: fields available for structured filters with `video.query()`.

In [5]:
CUSTOM_INDEX_NAME = f"custom_annotations_{uuid4().hex[:8]}"
INDEX_READY_STATUSES = {"ready", "done"}
INDEX_ACTIVE_STATUSES = {"building", "processing"}


def wait_until_index_ready(video, index, timeout=1800, poll_interval=10):
    deadline = time.time() + timeout

    while time.time() < deadline:
        latest_index = video.get_index(index_id=index.index_id)
        status = latest_index.status

        if status in INDEX_READY_STATUSES:
            return latest_index

        if status not in INDEX_ACTIVE_STATUSES:
            raise RuntimeError(f"Index build ended with status: {status}")

        time.sleep(poll_interval)

    raise TimeoutError(f"Index was not ready within {timeout} seconds.")


custom_annotation_index = video.index(
    name=CUSTOM_INDEX_NAME,
    source=custom_annotations,
    use_for=["semantic", "query"],
    fields={
        "semantic": ["description"],
        "text": ["description"],
        "filter": ["description", "annotation_type", "source"],
    },
)

custom_annotation_index = wait_until_index_ready(video, custom_annotation_index)

print("Custom annotation index:", custom_annotation_index.index_id, custom_annotation_index.status)
print("Index fields:", custom_annotation_index.fields)

Custom annotation index: 71f4fd612c0e4967 ready
Index fields: {'filter': ['description', 'annotation_type', 'source'], 'semantic': ['description'], 'text': ['description']}


## 6. Query Custom Annotations

Now query the custom index for annotations whose description contains `religious gathering`. The results include timestamped moments from the original video.

In [6]:
annotation_results = video.query(
    index_name=CUSTOM_INDEX_NAME,
    filter=[
        {
            "field": "description",
            "op": "contains",
            "value": "religious gathering",
        }
    ],
    limit=10,
    return_fields=["description", "annotation_type", "source"],
)

for shot in annotation_results.shots:
    metadata = getattr(shot, "metadata", {}) or {}
    print(f"{shot.start:.2f}s - {shot.end:.2f}s")
    print(metadata.get("description", ""))
    print("Annotation type:", metadata.get("annotation_type", ""))
    print("----")

10.00s - 100.00s
A religious gathering where people are praying and singing.
Annotation type: religious_gathering
----


## 7. Play the Matching Moments

Use the returned timestamps to generate a stream containing only the matching annotated moments.

In [7]:
result_timeline = [(shot.start, shot.end) for shot in annotation_results.shots]

if result_timeline:
    stream_link = video.generate_stream(result_timeline)
    player = play_stream(stream_link)
    display(player)
else:
    print("No matching annotations found.")

## Conclusion

You created a custom temporal annotation index from your own timestamped records and queried it with VideoDB.

This workflow is useful when you want to bring external metadata into VideoDB and make it searchable or filterable alongside the original media.

## Further Resources

- [Create an Index](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/indexing-pipelines/create-an-index)
- [Search and Retrieval](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/search-and-retrieval/natural-language-query)
